# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`

This notebook provides a guide for loading and exploring the [FAIR² dataset](https://sen.science/doi/10.71728/senscience.y7m0-f273) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(url)

# Print dataset metadata summary
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")
print(f"Identifier: {getattr(meta, 'identifier', None)}")
print(f"Published: {getattr(meta, 'datePublished', None)}")
print(f"License: {getattr(meta, 'license', None)}")


## 2. Data Overview
Review available record sets, fields, and their `@id`s.

All entities (record sets, fields, columns, etc) are referenced by their `@id` fields.

*Note*: The full schema (including record sets and fields) is accessible via `dataset.metadata`.


In [ ]:
# Show the available record sets in the dataset by @id
if hasattr(dataset.metadata, 'recordSet') and dataset.metadata.recordSet:
    record_sets = dataset.metadata.recordSet
    if not isinstance(record_sets, list):
        record_sets = [record_sets]
    print("Available record sets and their @id's:")
    for rs in record_sets:
        print(f"- Record set name: {getattr(rs, 'name', None)} | @id: {getattr(rs, '@id', None)}")
        if hasattr(rs, 'field') and rs.field:
            fields = rs.field if isinstance(rs.field, list) else [rs.field]
            print("    Fields:")
            for f in fields:
                print(f"        - Field name: {getattr(f, 'name', None)} | @id: {getattr(f, '@id', None)} | dataType: {getattr(f, 'dataType', None)}")
else:
    # If dataset.metadata.recordSet is empty, attempt to extract record set ids using records() method
    print("No recordSet metadata found in the schema. Discovering available record sets using 'dataset.records()'...")
    discovered_ids = set()
    for rsid in dataset.record_sets:
        discovered_ids.add(rsid)
    for rsid in discovered_ids:
        print(f"- @id: {rsid}")


## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.

**Note:** All record set, field, and column references below use their `@id`s, as per the FAIR² Croissant specification.

You may need to inspect the printed record set IDs above to select which record sets to extract.

In [ ]:
# Retrieve all record set @id's from the dataset
record_set_ids = list(dataset.record_sets)
print("Record Set @id's available in this dataset:")
for i, rsid in enumerate(record_set_ids):
    print(f"  [{i}] {rsid}")

# Extract records from all record sets into DataFrames
dataframes = {}
for rsid in record_set_ids:
    records = list(dataset.records(record_set=rsid))
    df = pd.DataFrame(records)
    dataframes[rsid] = df
    print(f"Loaded {len(df)} records from record set @id: {rsid}")

# As an example, pick the first record set
if record_set_ids:
    rs0 = record_set_ids[0]
    print(f"\nColumns in record set @id {rs0}:")
    print(dataframes[rs0].columns.tolist())
    display(dataframes[rs0].head())


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records, normalizing numeric fields, or grouping by attributes.

Below, operations are performed referencing fields by their `@id`.

In [ ]:
# Identify a numeric field (by @id) to analyze from the first DataFrame

# Inspect sample data and choose numeric columns
example_df = dataframes[rs0]
numeric_columns = example_df.select_dtypes(include='number').columns.tolist()
print(f"Numeric columns in record set {rs0}: {numeric_columns}")

# If there is at least one numeric field, use it for EDA
if numeric_columns:
    numeric_field_id = numeric_columns[0]
    group_candidates = [col for col in example_df.columns if example_df[col].dtype == 'object']
    group_field = group_candidates[0] if group_candidates else None
    threshold = example_df[numeric_field_id].mean() if example_df[numeric_field_id].notnull().any() else 0
    print(f"\nAnalyzing field (by @id): {numeric_field_id}\n")
    filtered_df = example_df[example_df[numeric_field_id] > threshold]
    print(f"Filtered records where {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())
    # Normalize numeric field
    filtered_df = filtered_df.copy()
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        (filtered_df[numeric_field_id].std(ddof=0) or 1)
    )
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    # Optionally group by a categorical field (also referenced by @id)
    if group_field:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"\nGrouped normalized data by {group_field}:")
        display(grouped_df.head())
else:
    print("No numeric fields detected in the selected record set.")


## 5. Visualization
Visualize the distribution and relationships in the selected record set using field and record set `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the numeric field's distribution
if numeric_columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(example_df[numeric_field_id].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of '{numeric_field_id}' in record set {rs0}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If grouped_df exists, plot means per group
    if group_field is not None and 'grouped_df' in locals():
        plt.figure(figsize=(10, 5))
        grouped_df[numeric_field_id].plot(kind='bar')
        plt.title(f"Mean of {numeric_field_id} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.tight_layout()
        plt.show()


## 6. Conclusion
In this notebook, we demonstrated how to load, inspect, and analyze a FAIR² Croissant dataset using `mlcroissant`, always referencing all data entities by their `@id`.

- We reviewed the dataset's metadata, record sets, and fields.
- We extracted the available data into DataFrames, using record set and field `@id`s.
- We filtered and normalized a numeric field, optionally grouping by a categorical attribute also referenced by `@id`.
- We visualized data distributions and group differences.

This structure can be adapted for any dataset described by a Croissant schema. For further analysis, always use entity `@id`s found in the schema or via programmatic inspection.
